# Anivora — Bulk Episode Upload (Google Colab)

Upload video episode dari Google Drive langsung ke Bunny Stream lewat protokol TUS (resumable). Internet Colab cepat, dan upload **tidak** lewat origin/Cloudflare jadi tidak kena limit 100MB.

**Cara kerja:** login admin ke Anivora → pilih anime + season → scan folder Drive → cocokkan file ke nomor episode → minta tiket upload (`createEpisodeUpload`) → upload langsung ke Bunny.

API key Bunny tidak pernah masuk ke notebook ini — server yang menandatangani tiket.

Jalankan cell berurutan dari atas.

In [ ]:
#@title 1. Install dependencies
!pip -q install tuspy tqdm requests

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Konfigurasi
from getpass import getpass

BASE_URL     = 'https://anime.mynekomy.site'  #@param {type:'string'}
ADMIN_EMAIL  = 'admin@example.com'            #@param {type:'string'}
# Folder Drive berisi file video episode (urutkan sesuai episode):
DRIVE_FOLDER = '/content/drive/MyDrive/anivora'  #@param {type:'string'}
# Lewati episode yang sudah punya video (jangan upload ulang):
SKIP_UPLOADED = True  #@param {type:'boolean'}

ADMIN_PASSWORD = getpass('Admin password: ')
print('OK. BASE_URL =', BASE_URL)

In [ ]:
#@title 4. Login + helper RPC
import requests

session = requests.Session()
session.headers.update({'content-type': 'application/json'})

def login():
    r = session.post(f'{BASE_URL}/api/auth/sign-in/email',
                     json={'email': ADMIN_EMAIL, 'password': ADMIN_PASSWORD})
    r.raise_for_status()
    me = session.post(f'{BASE_URL}/rpc/me', json={'json': {}})
    me.raise_for_status()
    user = me.json()['json']['user']
    if user.get('role') != 'admin':
        raise SystemExit('Akun ini bukan admin.')
    print('Login sebagai', user['email'], '(admin)')

def rpc(path, payload=None):
    r = session.post(f'{BASE_URL}/rpc/{path}', json={'json': payload or {}})
    data = r.json()
    if not r.ok:
        body = data.get('json', data)
        raise RuntimeError(f'{path} -> {r.status_code}: {body.get("message", body)}')
    return data['json']

login()

In [ ]:
#@title 5. Pilih anime
animes = rpc('admin/listAllAnime')['anime']
for i, a in enumerate(animes):
    print(f"[{i}] {a['title']}  ({a.get('status')})  id={a['id']}")

ANIME_INDEX = 0  #@param {type:'integer'}
anime = animes[ANIME_INDEX]
print('\nDipilih:', anime['title'])

In [ ]:
#@title 6. Pilih / buat season
seasons = rpc('admin/listSeasons', {'animeId': anime['id']})['seasons']
for i, s in enumerate(seasons):
    print(f"[{i}] Season {s['seasonNumber']} · {s.get('title') or '-'}  id={s['id']}")
if not seasons:
    print('(belum ada season)')

SEASON_INDEX  = 0  #@param {type:'integer'}
CREATE_SEASON = False  #@param {type:'boolean'}
NEW_SEASON_NUMBER = 1  #@param {type:'integer'}

if CREATE_SEASON:
    season = rpc('admin/createSeason',
                 {'animeId': anime['id'], 'seasonNumber': NEW_SEASON_NUMBER})['season']
    print('Season dibuat:', season['seasonNumber'])
else:
    season = seasons[SEASON_INDEX]
print('\nSeason aktif:', season['seasonNumber'], 'id=', season['id'])

In [ ]:
#@title 7. Scan folder Drive + cocokkan ke nomor episode
import os, re

VIDEO_EXT = ('.mp4', '.mkv', '.mov', '.webm', '.avi', '.m4v')

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

def episode_number(name):
    m = re.search(r'[Ee]p?(?:isode)?[ ._-]?(\d{1,3})', name)
    if not m:
        m = re.search(r'(?<!\d)(\d{1,3})(?!\d)', name)
    return int(m.group(1)) if m else None

files = [f for f in sorted(os.listdir(DRIVE_FOLDER), key=natural_key)
         if f.lower().endswith(VIDEO_EXT)]

# Auto-detect nomor; kalau tidak terdeteksi, pakai urutan (1,2,3,...)
mapping = []
for idx, f in enumerate(files, start=1):
    n = episode_number(f)
    mapping.append({'number': n if n is not None else idx, 'file': f})

print(f'{len(files)} file video ditemukan di {DRIVE_FOLDER}:\n')
for m in mapping:
    print(f"  E{m['number']:02d}  <-  {m['file']}")
print('\nPeriksa pemetaan di atas. Kalau nomor salah, rename file di Drive lalu ulangi cell ini.')

In [ ]:
#@title 8. Pastikan episode ada (bulk-create bila kurang)
episodes = rpc('admin/listEpisodes', {'seasonId': season['id']})['episodes']
have = {e['episodeNumber'] for e in episodes}
need_max = max((m['number'] for m in mapping), default=0)
cur_max = max(have, default=0)

if need_max > cur_max:
    count = need_max - cur_max
    res = rpc('admin/bulkCreateEpisodes', {'seasonId': season['id'], 'count': count})
    print(f"Bulk-create: {res['created']} episode dibuat (E{cur_max+1:02d}..E{need_max:02d}).")
    episodes = rpc('admin/listEpisodes', {'seasonId': season['id']})['episodes']
else:
    print('Semua episode yang dibutuhkan sudah ada.')

by_number = {e['episodeNumber']: e for e in episodes}
print('Total episode di season ini:', len(episodes))

In [ ]:
#@title 9. Upload semua file ke Bunny (TUS resumable)
import mimetypes
from tusclient import client as tus_client
from tqdm.auto import tqdm

def upload_one(path, ticket, title):
    mime = mimetypes.guess_type(path)[0] or 'video/mp4'
    headers = {
        'AuthorizationSignature': ticket['signature'],
        'AuthorizationExpire': str(ticket['expiration']),
        'VideoId': ticket['videoId'],
        'LibraryId': str(ticket['libraryId']),
    }
    cl = tus_client.TusClient(ticket['endpoint'], headers=headers)
    up = cl.uploader(path, chunk_size=64 * 1024 * 1024,
                     metadata={'filetype': mime, 'title': title})
    size = up.get_file_size()
    with tqdm(total=size, unit='B', unit_scale=True, desc=title) as bar:
        while up.offset < size:
            up.upload_chunk()
            bar.update(up.offset - bar.n)

done, skipped, failed = 0, 0, []
for m in mapping:
    ep = by_number.get(m['number'])
    if not ep:
        print(f"E{m['number']:02d}: episode tidak ada, dilewati"); skipped += 1; continue
    if SKIP_UPLOADED and ep.get('bunnyVideoId'):
        print(f"{ep['episodeCode']}: sudah punya video, dilewati"); skipped += 1; continue
    path = os.path.join(DRIVE_FOLDER, m['file'])
    try:
        ticket = rpc('admin/createEpisodeUpload', {'episodeId': ep['id']})
        upload_one(path, ticket, ep['episodeCode'])
        done += 1
    except Exception as e:
        print(f"{ep['episodeCode']}: GAGAL — {e}"); failed.append(ep['episodeCode'])

print(f'\nSelesai. Upload: {done}, dilewati: {skipped}, gagal: {len(failed)}')
if failed:
    print('Gagal:', ', '.join(failed))
print('Bunny butuh beberapa menit untuk memproses. Status episode akan jadi "ready" otomatis (poller server), atau klik Sync status di admin.')

## Catatan

- **Penamaan file**: usahakan ada nomor episode di nama file (mis. `...-01_720p.mp4`, `Ep 02`, `E03`). Kalau tidak terdeteksi, notebook pakai urutan abjad-natural.
- **Resume**: TUS resumable; kalau koneksi putus, jalankan ulang cell 9 — episode yang sudah ada video dilewati (jika `SKIP_UPLOADED`).
- **Status**: setelah upload, Bunny memproses video. Server punya poller yang menyetel status ke `ready`. Publish manual dari admin bila perlu.
- **Aman**: hanya butuh kredensial admin Anivora. API key Bunny tetap di server.